Outlining a feedforward neural network without Pytorch, no TensorFlow
The point is to implement some of these structures to develop stronger intuitions and understanding of these larger paradigms

In [37]:
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

In [ ]:
class Linear:
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * 0.01
        self.b = np.zeros(out_features)

    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, dout):
        self.dW = self.X.T @ dout
        self.db = dout.sum(axis=0)
        return dout @ self.W.T

class ReLU:
    def forward(self, X):
        self.X = X
        return np.maximum(0, X)

    def backward(self, dout):
        return np.where(self.X > 0, 1, 0) * dout

class SoftmaxCrossEntropy:
    def forward(self, X, y):
        self.y = y
        self.batch_size = X.shape[0]
        exps = np.exp(X - np.max(X, axis=1, keepdims=True))
        self.probs = exps / np.sum(exps, axis=1, keepdims=True)

        class_probability = self.probs[np.arange(X.shape[0]), y]
        return np.mean(-1 * np.log(class_probability))

    
    def backward(self):
        one_hot = np.zeros((self.batch_size, 10))
        one_hot[np.arange(self.batch_size), self.y] = 1
        return (self.probs - one_hot) / self.batch_size

In [39]:
mnist = fetch_openml('mnist_784', version=1)
X, y = mnist.data.to_numpy(), mnist.target.astype(int).to_numpy()

Here we preprocess the data

In [41]:
print(X.shape)
print(y.shape)
print(X.min(), X.max())

(70000, 784)
(70000,)
0 255


In [43]:
X = X / 255.0

In [44]:
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

Now we actally start the training loop

In [67]:
epochs = 100
batch_size = 32
linear_1 = Linear(784, 128)
linear_2 = Linear(128, 10)
relu = ReLU()
softmaxCE = SoftmaxCrossEntropy()
lr = 0.05
for epoch in range(epochs):
    epoch_loss = 0
    for i in range(0, 60000, batch_size):
        # Slice up data into batch size
        X_slice = X_train[i:i + batch_size]
        y_slice = y_train[i:i + batch_size]

        # Forward
        forward_linear_result = linear_1.forward(X_slice)
        relu_forward = relu.forward(forward_linear_result)
        forward_linear_2_result = linear_2.forward(relu_forward)
        loss = softmaxCE.forward(forward_linear_2_result, y_slice)

        # Update Epoch_Loss
        epoch_loss += loss

        # Backward
        backward_softmaxCE = softmaxCE.backward()
        backward_linear_2_result = linear_2.backward(backward_softmaxCE)
        relu_backward = relu.backward(backward_linear_2_result)
        backward_linear_1_result = linear_1.backward(relu_backward)

        # Update weights
        linear_1.W -= lr * linear_1.dW
        linear_1.b -= lr * linear_1.db
        linear_2.W -= lr * linear_2.dW
        linear_2.b -= lr * linear_2.db
        
    print(f"Loss: {epoch_loss}")







Loss: 4315.545036241884
Loss: 4315.350656896039
Loss: 4315.296497853532
Loss: 4315.225094319992
Loss: 4315.127245537119
Loss: 4314.990467562276
Loss: 4314.796179080175
Loss: 4314.517445626399
Loss: 4314.114760354437
Loss: 4313.530063424025
Loss: 4312.677724216483
Loss: 4311.431311337937
Loss: 4309.605450766233
Loss: 4306.9264491949025
Loss: 4302.992736806688
Loss: 4297.218606823956
Loss: 4288.755807682454
Loss: 4276.398593718584
Loss: 4258.460066735659
Loss: 4232.681569910076
Loss: 4196.1995845259335
Loss: 4145.74206598874
Loss: 4078.173080551558
Loss: 3991.4863105702334
Loss: 3885.902617951198
Loss: 3764.2566710944666
Loss: 3631.154752781076
Loss: 3491.5410260206186
Loss: 3349.913231091336
Loss: 3210.0766184057597
Loss: 3074.8879116942458
Loss: 2945.992867997847
Loss: 2823.9737602923115
Loss: 2708.815782663062
Loss: 2600.4172268319585
Loss: 2498.8632526106667
Loss: 2404.347970759813
Loss: 2316.998523534019
Loss: 2236.7309977185696
Loss: 2163.182750049909
Loss: 2095.821034429218
Loss: 